# Day 3 · 실습 ⑤ — DINOv3 Image Embedding과 유사도 검색

**연계 이론:** Day 3 이론 3(Self-Supervised Learning과 DINOv3)  
**모델:** `facebook/dinov3-vits16-pretrain-lvd1689m` (ViT-S/16, 384차원 표현)

### 실습 목표
1. 사전학습 DINOv3 모델로 이미지 특징 추출
2. 모델 입력과 `CLS / Register / Patch` Token Shape 관찰
3. 여러 이미지의 Embedding을 L2 정규화하고 Cosine Similarity 계산
4. 유사도 Heatmap과 Top-k 유사 이미지 검색 결과 확인

**50분 운영 예시:** 입력·모델 구조 10분 → 특징 추출 10분 → 유사도·Heatmap 15분 → 검색 10분 → 정리 5분.

> **중요:** 이 모델의 공식 Hugging Face 가중치는 *접근 승인(Gated)*이 필요합니다. 승인 없이 실행되는 것처럼 구성하지 않았습니다. 강사에게 승인된 모델 파일을 제공받거나 계정별 승인을 완료한 뒤 실행하세요.

## 0. 모델 접근과 사전 준비

1. [DINOv3 ViT-S/16 모델 페이지](https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m)에서 라이선스·접근 조건을 확인하고 **본인 계정으로 접근 승인**을 받습니다.
2. 교육기관이 승인된 가중치를 적법하게 로컬에 배포했다면 `DAY3_DINOV3_MODEL` 환경변수에 **모델 폴더 경로**(예: `.../models/dinov3-vits16`)를 넣습니다. 모델 폴더에는 `config.json`, `preprocessor_config.json`, 가중치 파일 등이 필요합니다.
3. Hugging Face에서 직접 받을 경우 터미널에서 `hf auth login` 후 개인 계정으로 인증합니다. **Notebook에 토큰 문자열을 붙여 넣거나 공동 계정을 공유하지 않습니다.**
4. `transformers>=4.56.0`, `torch`, `torchvision`, `huggingface_hub`, `pillow`, `numpy`, `pandas`, `matplotlib`가 필요합니다.
5. 이미지 데이터는 PyTorch 공식 개미·벌 검증 이미지 12장을 사용합니다. 앞선 실습을 실행했다면 데이터가 재사용됩니다.

공식 참고: [Meta DINOv3](https://github.com/facebookresearch/dinov3), [Transformers DINOv3 문서](https://huggingface.co/docs/transformers/model_doc/dinov3).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = os.getenv("DAY3_DINOV3_MODEL", "facebook/dinov3-vits16-pretrain-lvd1689m")
print("torch:", torch.__version__, "| device:", DEVICE)
print("DINOv3 source:", MODEL_ID)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. 비교할 이미지 선택

PyTorch 검증 데이터(`ants`, `bees`)에서 클래스별 6장씩 **고정된 샘플**을 선택합니다. 같은 데이터셋을 계속 사용하는 이유는 앞선 ResNet 실습과 연결하기 위해서입니다. 데이터 선택에는 정답 클래스가 있지만, **DINOv3 특징 추출에는 정답 레이블을 전달하지 않습니다.**

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA_ROOT = Path(os.getenv("DAY3_DATA_ROOT", str(Path.cwd() / "data" / "hymenoptera_data"))).expanduser().resolve()
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

def ensure_dataset(root=DATA_ROOT):
    required = [root / "train" / "ants", root / "train" / "bees",
                root / "val" / "ants", root / "val" / "bees"]
    if all(p.is_dir() for p in required):
        print(f"데이터셋 준비됨: {root}")
        return root

    # PyTorch 공식 튜토리얼의 공개 개미/벌 데이터셋을 1회 내려받습니다.
    # 공동 실습에서는 강사가 미리 다운로드하여 DAY3_DATA_ROOT를 지정하세요.
    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "hymenoptera_data.zip"
    if not archive.is_file():
        print("공식 PyTorch 개미/벌 데이터셋 다운로드 중...")
        tmp_file = archive.with_suffix(".zip.part")
        try:
            with urllib.request.urlopen(DATA_URL, timeout=90) as response, tmp_file.open("wb") as out:
                shutil.copyfileobj(response, out)
            tmp_file.replace(archive)
        finally:
            tmp_file.unlink(missing_ok=True)

    with zipfile.ZipFile(archive) as zf:
        target_base = root.parent.resolve()
        for entry in zf.infolist():
            dest = (target_base / entry.filename).resolve()
            if not dest.is_relative_to(target_base):
                raise ValueError(f"압축 경로가 허용 범위를 벗어났습니다: {entry.filename}")
        zf.extractall(target_base)

    if not all(p.is_dir() for p in required):
        raise FileNotFoundError(f"데이터셋의 train/val/ants/bees 폴더를 찾지 못했습니다: {root}")
    print(f"데이터셋 다운로드 완료: {root}")
    return root

ensure_dataset()

In [ ]:
# [학생이 변경할 수 있는 설정] 클래스당 이미지 수
IMAGES_PER_CLASS = 6
rng = np.random.default_rng(SEED)

image_paths, true_labels = [], []
for class_name in ("ants", "bees"):
    available = sorted((DATA_ROOT / "val" / class_name).glob("*.jpg"))
    if len(available) < IMAGES_PER_CLASS:
        raise ValueError(f"{class_name}: 이미지가 {IMAGES_PER_CLASS}장보다 적습니다.")
    choices = rng.choice(len(available), size=IMAGES_PER_CLASS, replace=False)
    for idx in choices:
        image_paths.append(available[int(idx)])
        true_labels.append(class_name)

images = []
for path in image_paths:
    with Image.open(path) as im:
        images.append(im.convert("RGB"))
short_labels = [f"{name}-{i+1}" for name in ("ants", "bees")
                for i in range(IMAGES_PER_CLASS)]

fig, axes = plt.subplots(2, IMAGES_PER_CLASS, figsize=(15, 5.5))
for ax, im, label in zip(axes.flat, images, short_labels):
    ax.imshow(im)
    ax.set_title(label)
    ax.axis("off")
plt.suptitle("Fixed validation images (no labels sent to DINOv3)")
plt.tight_layout()
plt.show()
print("선택한 이미지 수:", len(images))

## 2. DINOv3 사전학습 모델 불러오기

`AutoImageProcessor`는 모델에 맞는 이미지 전처리를, `AutoModel`은 사전학습 Backbone을 불러옵니다. 다운로드에는 가중치 접근 권한과 네트워크가 필요합니다. 수업에서는 강사가 미리 모델 파일을 캐시해 두는 방법을 권장합니다.

In [ ]:
try:
    processor = AutoImageProcessor.from_pretrained(MODEL_ID)
    model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
except Exception as exc:
    raise RuntimeError(
        "DINOv3 모델을 불러오지 못했습니다. 공식 모델 페이지에서 접근 승인을 받고 "
        "`hf auth login`으로 본인 계정에 로그인하거나, 승인된 로컬 모델 폴더를 "
        "DAY3_DINOV3_MODEL에 지정하세요. 인터넷/캐시/transformers 버전도 확인하세요. "
        "원래 오류 내용은 아래 원인(traceback)을 참고하세요."
    ) from exc

print("model type:", model.__class__.__name__)
print("patch_size:", model.config.patch_size)
print("hidden_size:", model.config.hidden_size)
print("register tokens:", model.config.num_register_tokens)

## 3. 이미지 → Token → Global Embedding

DINOv3 ViT-S/16에서 `last_hidden_state`는 일반적으로 `CLS Token + Register Token + Patch Token` 순서입니다. 이미지 전체를 비교할 이번 실습에서는 **첫 번째 CLS Token**을 Global Image Embedding으로 사용합니다. 이미지 크기에 따라 Patch Token 수는 달라지므로 Shape를 코드로 계산해 확인합니다.

In [ ]:
# 배치 추론: 전체 이미지를 한 번에 GPU 메모리에 올리지 않습니다.
INFERENCE_BATCH_SIZE = 4
all_cls_vectors = []
first_token_shape = None
for start in range(0, len(images), INFERENCE_BATCH_SIZE):
    batch = images[start:start + INFERENCE_BATCH_SIZE]
    inputs = processor(images=batch, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model(**inputs)
    hidden = outputs.last_hidden_state  # [B, CLS+registers+patches, D]
    if first_token_shape is None:
        first_token_shape = tuple(hidden.shape)
        _, _, h, w = inputs["pixel_values"].shape
        patch_size = model.config.patch_size
        if isinstance(patch_size, (tuple, list)):
            patch_h, patch_w = patch_size
        else:
            patch_h = patch_w = patch_size
        num_patch = (h // patch_h) * (w // patch_w)
        num_register = model.config.num_register_tokens
        expected_tokens = 1 + num_register + num_patch
        print("pixel_values:", tuple(inputs["pixel_values"].shape))
        print("tokens = CLS(1) + register", num_register, "+ patch", num_patch,
              "=", expected_tokens)
        print("last_hidden_state:", first_token_shape)
        assert hidden.shape[1] == expected_tokens
    cls_vectors = hidden[:, 0, :]  # Global image embedding
    all_cls_vectors.append(cls_vectors.detach().float().cpu())

embeddings = torch.cat(all_cls_vectors, dim=0)  # [N, D]
normalized = F.normalize(embeddings, dim=1)
print("Image Embeddings Shape:", tuple(embeddings.shape))
print("Normalized vector norm (first):", round(normalized[0].norm().item(), 6))

## 4. Cosine Similarity 행렬과 Heatmap

L2 정규화한 벡터끼리의 내적은 Cosine Similarity와 같습니다. 계산된 행렬의 **대각선은 자기 자신과의 비교**여서 약 1이 됩니다. 나머지 값이 이미지 사이의 유사도를 나타냅니다.

In [ ]:
similarity = (normalized @ normalized.T).numpy()
assert similarity.shape == (len(images), len(images))
assert np.allclose(np.diag(similarity), 1.0, atol=1e-5)

similarity_df = pd.DataFrame(similarity, index=short_labels, columns=short_labels)
display(similarity_df.round(3))

fig, ax = plt.subplots(figsize=(8.5, 7))
heatmap = ax.imshow(similarity, vmin=-1.0, vmax=1.0, cmap="viridis")
ax.set_xticks(range(len(short_labels)), short_labels, rotation=65, ha="right")
ax.set_yticks(range(len(short_labels)), short_labels)
ax.set_title("DINOv3 CLS Embedding Cosine Similarity")
fig.colorbar(heatmap, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## 5. Top-k 이미지 검색

**조건 변경 활동(B):** `QUERY_INDEX`와 `TOP_K`를 변경해 보세요. 검색 기준 이미지 **자기 자신은 제외**합니다. 유사도가 높은 결과가 같은 클래스인지, 다른 시각적 특징에 영향을 받았는지 관찰합니다.

In [ ]:
# [학생이 변경할 수 있는 설정]
QUERY_INDEX = 0  # 0 ~ len(images)-1
TOP_K = 3
if not 0 <= QUERY_INDEX < len(images):
    raise ValueError("QUERY_INDEX가 범위를 벗어났습니다.")
if not 1 <= TOP_K < len(images):
    raise ValueError("TOP_K는 1 이상, 이미지 개수 미만이어야 합니다.")

query_scores = similarity[QUERY_INDEX].copy()
query_scores[QUERY_INDEX] = -np.inf  # 자기 자신 제외
ranked = np.argsort(query_scores)[::-1][:TOP_K]

fig, axes = plt.subplots(1, TOP_K + 1, figsize=(3.5 * (TOP_K + 1), 3.5))
axes[0].imshow(images[QUERY_INDEX])
axes[0].set_title(f"Query: {short_labels[QUERY_INDEX]}")
axes[0].axis("off")
for rank, (idx, ax) in enumerate(zip(ranked, axes[1:]), start=1):
    ax.imshow(images[idx])
    ax.set_title(f"Top {rank}: {short_labels[idx]}\ncos={similarity[QUERY_INDEX, idx]:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("검색 결과 (정답 레이블은 확인용):")
for rank, idx in enumerate(ranked, start=1):
    print(f"{rank}위: {short_labels[idx]} / cos={similarity[QUERY_INDEX, idx]:.4f} / "
          f"class={true_labels[idx]}")

## 6. 결과 저장과 핵심 질문

다음 파일을 `results/`에 저장합니다. 모델 추론에 사용한 CLS 벡터와 유사도 행렬이며, 분류 Head를 학습한 결과는 아닙니다.

In [ ]:
result_dir = Path.cwd() / "results"
result_dir.mkdir(parents=True, exist_ok=True)
np.save(result_dir / "day3_03_dinov3_cls_embeddings.npy", embeddings.numpy())
similarity_df.to_csv(result_dir / "day3_03_dinov3_similarity.csv", encoding="utf-8-sig")
print("저장 완료:", result_dir)

**학습 내용 확인**

1. DINOv3는 특징을 추출하는 과정에서 왜 정답 클래스를 입력받지 않나요?
2. CLS Token과 Patch Token은 어떤 차이가 있나요?
3. Embedding 벡터를 L2 정규화하는 이유는 무엇인가요?
4. 유사도 행렬에서 대각선이 약 1인 이유는 무엇인가요?
5. `QUERY_INDEX`를 바꾸면 Top-k 결과는 어떻게 달라지나요?

**다음 학습 연결:** DINOv3는 이미지를 이미지와 비교했습니다. 다음에는 CLIP으로 **이미지와 텍스트**를 비교하여 Zero-shot 분류를 수행합니다.